# xG-Projekt – 01: Datenbeschaffung & Feature Engineering

**Datensatz:** StatsBomb Open Data – 1. Bundesliga 2023/2024 (34 Spiele)

Dieses Notebook:
1. lädt alle Spiele der Saison via `statsbombpy`
2. definiert und erklärt die Feature-Engineering-Funktionen (Distanz, Winkel, Freeze-Frame-Analyse)
3. wendet sie auf alle Schuss-Events an
4. speichert das Ergebnis als `data/shots.csv` für die weiteren Notebooks (EDA, Modellierung)

Die Feature-Funktionen sind hier vollständig sichtbar implementiert. Identischer Code liegt
zusätzlich in `src/features.py`, damit er auch außerhalb von Notebooks (z.B. in
`src/build_dataset.py` für einen schnellen Re-Run) wiederverwendbar ist.


## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import math
import time
import pandas as pd
import numpy as np
from statsbombpy import sb


## 1. Datenbeschaffung

StatsBomb Open Data ist frei zugänglich (keine Anmeldung nötig), Lizenz erlaubt akademische
Nutzung mit Attribution. Wir verwenden die 1. Bundesliga 2023/2024 (`competition_id=9`,
`season_id=281`): 34 Spiele mit vollständigen Freeze-Frame-Daten (Spielerpositionen im
Moment jedes Schusses).

**Kritische Einordnung des Datensatzes** (relevant für die Präsentation):
- Event-Tagging durch StatsBomb-Analysten -> menschliche Fehlerquelle bei Koordinaten möglich
- Nur eine Liga/Saison -> Ergebnisse ggf. nicht 1:1 auf andere Ligen/Spielklassen übertragbar
- Klassenungleichgewicht: nur ein Bruchteil aller Schüsse sind Tore -> wichtig für Metrik-Wahl
  später (nicht nur Accuracy!)


In [2]:
COMPETITION_ID = 9   # 1. Bundesliga
SEASON_ID = 281      # 2023/2024

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
print(f"{len(matches)} Spiele gefunden")
matches[["match_id", "match_date", "home_team", "away_team", "home_score", "away_score"]].head()


34 Spiele gefunden


,match_id,match_date,home_team,away_team,home_score,away_score
0,3895292,2024-04-06,Union Berlin,Bayer Leverkusen,0,1
1,3895320,2024-04-27,Bayer Leverkusen,VfB Stuttgart,2,2
2,3895158,2023-12-03,Bayer Leverkusen,Borussia Dortmund,1,1
3,3895107,2023-10-08,Bayer Leverkusen,FC Köln,3,0
4,3895340,2024-05-12,Bochum,Bayer Leverkusen,0,5


## 2. Feature-Engineering-Funktionen

StatsBomb-Koordinatensystem: Spielfeld 120 x 80 (Länge x Breite). Das gegnerische Tor liegt
bei x=120, Mitte bei y=40, Torpfosten bei y=36 und y=44.

### 2.1 Distanz zum Tor

Je näher der Schütze am Tor steht, desto wahrscheinlicher ein Treffer – die einfachste und
wichtigste Kennzahl jedes xG-Modells.


In [3]:
GOAL_X = 120.0
GOAL_Y = 40.0
GOAL_POST_1 = 36.0
GOAL_POST_2 = 44.0


def distance_to_goal(location):
    """Euklidische Distanz vom Schussort zur Torlinienmitte."""
    x, y = location[0], location[1]
    return math.hypot(GOAL_X - x, GOAL_Y - y)


### 2.2 Sichtwinkel auf das Tor

Ein Schuss aus der Mitte hat einen größeren Sichtwinkel auf die Torbreite als ein Schuss aus
spitzem Winkel von der Seitenlinie – auch bei gleicher Distanz macht das einen großen
Unterschied in der Torwahrscheinlichkeit. Klassisches Kernfeature jedes echten xG-Modells
(StatsBomb, Opta etc.).


In [4]:
def angle_to_goal(location):
    """
    Sichtwinkel (in Grad) auf die Torbreite vom Schussort aus.
    Je größer der Winkel, desto größer die 'sichtbare' Torfläche.
    """
    x, y = location[0], location[1]
    dx = GOAL_X - x
    dy1 = GOAL_POST_1 - y
    dy2 = GOAL_POST_2 - y
    if dx <= 0:
        return 0.0
    angle1 = math.atan2(dy1, dx)
    angle2 = math.atan2(dy2, dx)
    return abs(math.degrees(angle2 - angle1))


### 2.3 Gegenspieler im Schusskegel (aus Freeze-Frame)

StatsBomb liefert für jeden Schuss ein "Freeze Frame": die Positionen aller sichtbaren Spieler
im exakten Moment des Schusses. Daraus berechnen wir, wie viele Gegner sich zwischen Schütze
und Tor im direkten Sichtkegel befinden – ein starkes Signal dafür, wie "zugestellt" ein Schuss
war. Das ist der anspruchsvollste Teil unseres Feature Engineerings.


In [5]:
def defenders_in_cone(location, freeze_frame):
    """
    Zählt gegnerische Spieler, die sich näher am Tor als am Schützen befinden UND
    innerhalb des Sichtkegels zum Tor liegen (Winkeltoleranz 8 Grad).
    Grobe, aber gängige Approximation für 'wie zugestellt ist der Schuss'.
    """
    if not isinstance(freeze_frame, list):
        return None
    x, y = location[0], location[1]
    count = 0
    for p in freeze_frame:
        if p.get("teammate"):
            continue
        px, py = p["location"][0], p["location"][1]
        if px <= x:  # nur Spieler zwischen Schütze und Tor berücksichtigen
            continue
        shot_angle = math.atan2(GOAL_Y - y, GOAL_X - x)
        player_angle = math.atan2(py - y, px - x)
        if abs(math.degrees(shot_angle - player_angle)) < 8:
            count += 1
    return count


def goalkeeper_distance_to_goal(freeze_frame):
    """Distanz des gegnerischen Torwarts zur eigenen Torlinie."""
    if not isinstance(freeze_frame, list):
        return None
    for p in freeze_frame:
        if p.get("position", {}).get("name") == "Goalkeeper" and not p.get("teammate"):
            gx, gy = p["location"][0], p["location"][1]
            return math.hypot(GOAL_X - gx, GOAL_Y - gy)
    return None


### 2.4 Alles zusammenführen

Eine Funktion, die aus einer rohen Event-Zeile alle Features plus Zielvariable (`is_goal`)
extrahiert.


In [6]:
def extract_shot_features(shot_row):
    """Nimmt eine Zeile aus dem StatsBomb-Events-DataFrame (type == 'Shot')
    und gibt ein dict mit allen ML-Features zurück."""
    loc = shot_row["location"]
    freeze = shot_row.get("shot_freeze_frame")

    return {
        "match_id": shot_row["match_id"],
        "player": shot_row.get("player"),
        "team": shot_row.get("team"),
        "minute": shot_row.get("minute"),
        "x": loc[0],
        "y": loc[1],
        "distance_to_goal": distance_to_goal(loc),
        "angle_to_goal": angle_to_goal(loc),
        "body_part": shot_row.get("shot_body_part"),
        "shot_type": shot_row.get("shot_type"),
        "technique": shot_row.get("shot_technique"),
        "under_pressure": bool(shot_row.get("under_pressure", False)),
        "n_defenders_in_cone": defenders_in_cone(loc, freeze),
        "gk_distance_to_goal": goalkeeper_distance_to_goal(freeze),
        "play_pattern": shot_row.get("play_pattern"),
        "statsbomb_xg": shot_row.get("shot_statsbomb_xg"),  # nur zum Vergleich, kein Feature!
        "is_goal": 1 if shot_row.get("shot_outcome") == "Goal" else 0,
    }


**Hinweis:** Diese vier Funktionen sind 1:1 identisch in `src/features.py` hinterlegt, damit
`src/build_dataset.py` den Datensatz auch außerhalb dieses Notebooks (z.B. per Skript-Aufruf)
neu erzeugen kann, ohne Code zu duplizieren oder Notebooks parsen zu müssen.


## 3. Feature-Funktionen kurz testen

Bevor wir auf den ganzen Datensatz losgehen: ein Sanity Check mit zwei Beispiel-Positionen.


In [7]:
# Schuss aus zentraler Position nah am Tor
center_close = [110, 40]
# Schuss von der Seitenlinie, weiter weg
wide_far = [95, 75]

print("Zentral, nah:  Distanz =", round(distance_to_goal(center_close), 2),
      "| Winkel =", round(angle_to_goal(center_close), 1), "Grad")
print("Seitlich, weit: Distanz =", round(distance_to_goal(wide_far), 2),
      "| Winkel =", round(angle_to_goal(wide_far), 1), "Grad")


Zentral, nah:  Distanz = 10.0 | Winkel = 43.6 Grad
Seitlich, weit: Distanz = 43.01 | Winkel = 6.2 Grad


Wie erwartet: der zentrale, nahe Schuss hat sowohl geringere Distanz als auch deutlich
größeren Torwinkel -> beide Features zeigen in die gleiche, plausible Richtung.

## 4. Schuss-Events aus allen Spielen extrahieren


In [8]:
match_ids = matches["match_id"].tolist()
all_shots = []
errors = 0
start = time.time()

for i, mid in enumerate(match_ids):
    try:
        events = sb.events(match_id=mid)
        shots = events[events["type"] == "Shot"].copy()
        shots["match_id"] = mid
        for _, row in shots.iterrows():
            all_shots.append(extract_shot_features(row))
    except Exception as e:
        errors += 1
        print(f"  Fehler bei match_id {mid}: {e}")

    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        print(f"  {i+1}/{len(match_ids)} Spiele verarbeitet ({elapsed:.0f}s, {errors} Fehler)")

df = pd.DataFrame(all_shots)
print(f"\nFertig: {len(df)} Schüsse extrahiert")


  10/34 Spiele verarbeitet (9s, 0 Fehler)


  20/34 Spiele verarbeitet (16s, 0 Fehler)


  30/34 Spiele verarbeitet (26s, 0 Fehler)



Fertig: 916 Schüsse extrahiert


## 5. Feature-Übersicht

| Feature | Beschreibung |
|---|---|
| `distance_to_goal` | Distanz Schussort -> Tormitte |
| `angle_to_goal` | Sichtwinkel auf die Torbreite (Grad) |
| `body_part` | Kopf / linker Fuß / rechter Fuß |
| `shot_type` | Freistoß / Elfmeter / offenes Spiel |
| `under_pressure` | Schütze stand unter Druck |
| `n_defenders_in_cone` | Gegenspieler im Schusskegel (aus Freeze-Frame) |
| `gk_distance_to_goal` | Position des Torwarts relativ zum Tor |
| `statsbomb_xg` | StatsBombs eigener xG-Wert – **nur zum späteren Vergleich, kein Feature!** |
| `is_goal` | Zielvariable (1 = Tor, 0 = kein Tor) |


In [9]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 916 entries, 0 to 915
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   match_id             916 non-null    int64  
 1   player               916 non-null    str    
 2   team                 916 non-null    str    
 3   minute               916 non-null    int64  
 4   x                    916 non-null    float64
 5   y                    916 non-null    float64
 6   distance_to_goal     916 non-null    float64
 7   angle_to_goal        916 non-null    float64
 8   body_part            916 non-null    str    
 9   shot_type            916 non-null    str    
 10  technique            916 non-null    str    
 11  under_pressure       916 non-null    bool   
 12  n_defenders_in_cone  908 non-null    float64
 13  gk_distance_to_goal  908 non-null    float64
 14  play_pattern         916 non-null    str    
 15  statsbomb_xg         916 non-null    float64
 16  i

In [10]:
df.head(10)


,match_id,player,team,minute,x,y,distance_to_goal,angle_to_goal,body_part,shot_type,technique,under_pressure,n_defenders_in_cone,gk_distance_to_goal,play_pattern,statsbomb_xg,is_goal
0,3895292,Borja Iglesias Quintas,Bayer Leverkusen,2,107.0,53.4,18.669762,17.365796,Right Foot,Open Play,Half Volley,True,1.0,3.612478,Regular Play,0.084877,0
1,3895292,Granit Xhaka,Bayer Leverkusen,5,96.5,39.9,23.500213,19.319453,Right Foot,Open Play,Normal,True,2.0,2.220360,From Throw In,0.042399,0
2,3895292,Adam Hložek,Bayer Leverkusen,6,88.2,45.5,32.272124,13.932473,Right Foot,Open Play,Normal,True,3.0,2.433105,From Corner,0.017574,0
3,3895292,Alejandro Grimaldo García,Bayer Leverkusen,10,98.5,49.2,23.385679,17.951508,Left Foot,Free Kick,Normal,True,5.0,1.264911,From Free Kick,0.075614,0
4,3895292,Borja Iglesias Quintas,Bayer Leverkusen,14,111.2,37.8,9.070832,46.726567,Head,Open Play,Normal,True,1.0,1.902630,Regular Play,0.118389,0
5,3895292,Alejandro Grimaldo García,Bayer Leverkusen,27,107.5,39.1,12.532358,35.333521,Right Foot,Open Play,Normal,True,2.0,2.319483,Regular Play,0.119091,0
6,3895292,Rani Khedira,Union Berlin,30,96.5,33.7,24.329817,18.077868,Left Foot,Open Play,Normal,True,1.0,1.902630,Regular Play,0.048804,0
7,3895292,Robin Gosens,Union Berlin,35,112.4,31.4,11.476933,27.717712,Head,Open Play,Normal,True,1.0,1.749286,From Corner,0.061352,0
8,3895292,Piero Martín Hincapié Reyna,Bayer Leverkusen,37,109.3,36.4,11.289376,37.526332,Head,Open Play,Normal,True,2.0,1.421267,From Free Kick,0.047790,0
9,3895292,Yorbe Vertessen,Union Berlin,38,105.8,35.8,14.808106,29.197991,Right Foot,Open Play,Normal,True,2.0,2.200000,From Keeper,0.082499,0


## 6. Sanity Checks

In [11]:
print("Fehlende Werte pro Spalte:")
print(df.isna().sum())
print()
print(f"Tore: {df['is_goal'].sum()} von {len(df)} Schüssen ({df['is_goal'].mean()*100:.1f}%)")


Fehlende Werte pro Spalte:
match_id               0
player                 0
team                   0
minute                 0
x                      0
y                      0
distance_to_goal       0
angle_to_goal          0
body_part              0
shot_type              0
technique              0
under_pressure         0
n_defenders_in_cone    8
gk_distance_to_goal    8
play_pattern           0
statsbomb_xg           0
is_goal                0
dtype: int64

Tore: 110 von 916 Schüssen (12.0%)


In [12]:
# Plausibilitätscheck: Tore sollten im Schnitt aus geringerer Distanz und größerem Winkel fallen
df.groupby("is_goal")[["distance_to_goal", "angle_to_goal"]].mean()


,distance_to_goal,angle_to_goal
is_goal,,
0,18.790622,24.660300
1,13.074685,38.254362


Tore passieren im Schnitt aus deutlich geringerer Distanz als Nicht-Tore – ein gutes Zeichen,
dass die Features sinnvolles Signal enthalten. Die ausführliche EDA (Verteilungen, Shot Maps,
Korrelationen) folgt in Notebook 02.


## 7. Datensatz speichern

In [13]:
df.to_csv("../data/shots.csv", index=False)
print("Gespeichert: data/shots.csv")


Gespeichert: data/shots.csv
